# Preprocessing

---

1. Import Library dan Setup Folder

In [ ]:
import os
import rasterio
from rasterio.windows import Window
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Path menggunakan format relative yang aman di semua OS
BASE_DIR = Path("..") / "data"
RAW_DIR = BASE_DIR / "raw"
PATCH_OUT_DIR = BASE_DIR / "processed" / "patches_jabar"

# Buat folder output jika belum ada
PATCH_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Setup direktori selesai!")
print(f"📂 Folder Raw Data: {RAW_DIR}")
print(f"📂 Folder Output Patches: {PATCH_OUT_DIR}")

✅ Setup direktori selesai!
📂 Folder Raw Data: ..\data\raw
📂 Folder Output Patches: ..\data\processed\patches_jabar


2. Memeriksa Data Mentah (Raw Data)

In [ ]:
# Mencari semua file TIF di folder raw
tif_files = list(RAW_DIR.glob("Dataset_S2_OpenBuildings_Jabar_2025*.tif"))

print(f"🔍 Ditemukan {len(tif_files)} file pecahan GeoTIFF.")
for i, f in enumerate(tif_files):
    print(f"   [{i+1}] {f.name}")

if len(tif_files) == 0:
    raise FileNotFoundError("Peringatan: File .tif tidak ditemukan di folder /data/raw/ !")

🔍 Ditemukan 0 file pecahan GeoTIFF.


FileNotFoundError: Peringatan: File .tif tidak ditemukan di folder /data/raw/ !

3. Mesin Pemotong dan Pemberi Label Otomatis (The Core)

In [ ]:
PATCH_SIZE = 128
metadata_records = []

print("🚀 Memulai proses Slicing & Labeling...\n")

for file_idx, tif_path in enumerate(tif_files):
    with rasterio.open(tif_path) as src:
        height, width = src.height, src.width
        profile = src.profile
        
        total_steps = (height // PATCH_SIZE) * (width // PATCH_SIZE)
        
        with tqdm(total=total_steps, desc=f"Memproses {tif_path.name[:15]}") as pbar:
            for y in range(0, height, PATCH_SIZE):
                for x in range(0, width, PATCH_SIZE):
                    pbar.update(1)
                    
                    if y + PATCH_SIZE > height or x + PATCH_SIZE > width:
                        continue
                    
                    window = Window(x, y, PATCH_SIZE, PATCH_SIZE)
                    patch_data = src.read(window=window)
                    
                    sentinel_bands = patch_data[0:6, :, :] # 6 Band Sentinel
                    building_mask = patch_data[6, :, :]    # Band ke-7 Open Buildings
                    
                    # 1. FILTER NODATA (Buang jika > 20% area kosong)
                    if np.mean(sentinel_bands == 0) > 0.20:
                        continue
                    
                    # 2. HITUNG DENSITAS BANGUNAN
                    total_pixels = PATCH_SIZE * PATCH_SIZE
                    building_density = np.sum(building_mask == 1) / total_pixels
                    
                    # Buang jika area non-pemukiman (bangunan < 1%)
                    if building_density < 0.01: 
                        continue
                    
                    # 3. LABELING SLUM VS NON-SLUM
                    category = "slum" if building_density >= 0.65 else "non_slum"
                    split_assign = "train" if np.random.rand() < 0.8 else "val"
                    
                    # 4. SIMPAN PATCH (HANYA 6 BAND)
                    patch_name = f"patch_f{file_idx}_y{y}_x{x}.tif"
                    patch_path = PATCH_OUT_DIR / patch_name
                    
                    patch_profile = profile.copy()
                    patch_profile.update({
                        'count': 6, # Penting agar cocok dengan CNN 10-Channel kamu nanti
                        'height': PATCH_SIZE,
                        'width': PATCH_SIZE,
                        'transform': rasterio.windows.transform(window, src.transform)
                    })
                    
                    with rasterio.open(patch_path, 'w', **patch_profile) as dst:
                        dst.write(sentinel_bands)
                    
                    # 5. CATAT METADATA
                    metadata_records.append({
                        "patch_name": patch_name,
                        "patch_path": f"data/processed/patches_jabar/{patch_name}",
                        "category": category,
                        "split": split_assign,
                        "building_density": building_density
                    })

print("\n✅ Proses ekstraksi patch selesai!")

4. Visualisasi Imbalance & Distribusi Kepadatan Bangunan

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
# Load data sementara ke DataFrame untuk keperluan plotting
df_plot = pd.DataFrame(metadata_records)

# Setup canvas visualisasi (1 baris, 2 kolom)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- PLOT 1: CEK IMBALANCE KELAS ---
sns.countplot(data=df_plot, x='category', palette='Set2', ax=axes[0], order=['non_slum', 'slum'])
axes[0].set_title('Perbandingan Jumlah Kelas (Cek Imbalance)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Kategori Pemukiman', fontsize=10)
axes[0].set_ylabel('Jumlah Patch Gambar', fontsize=10)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# Tambahkan angka total di atas setiap batang grafik
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', 
                     xytext=(0, 8), 
                     textcoords='offset points', 
                     fontsize=10, fontweight='bold')

# --- PLOT 2: DISTRIBUSI KEPADATAN BANGUNAN ---
sns.histplot(data=df_plot, x='building_density', bins=30, kde=True, color='skyblue', ax=axes[1])
# Tambahkan garis threshold pemisah kelas slum
axes[1].axvline(x=0.65, color='red', linestyle='--', linewidth=2, label='Batas Threshold Slum (0.65)')
axes[1].set_title('Distribusi Kepadatan Bangunan se-Jawa Barat', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Rasio Kepadatan (0.0 = Kosong, 1.0 = Padat Total)', fontsize=10)
axes[1].set_ylabel('Frekuensi Kemunculan Patch', fontsize=10)
axes[1].legend()
axes[1].grid(linestyle='--', alpha=0.5)

# Tampilkan grafik keseluruhan
plt.tight_layout()
plt.show()

5. Export CSV dan Statistik Data

In [ ]:
# 1. Konversi seluruh hasil ekstraksi tanpa potongan ke DataFrame
df_final = pd.DataFrame(metadata_records)

print("================ RANGKUMAN DATASET UTUH (UNTUK TWO-STAGE) ================")
print(f"✅ Total Patch Terdaftar di CSV : {len(df_final)} patch")
print(f"📌 Komposisi Kelas Awal          :\n{df_final['category'].value_counts()}")
print(f"📈 Distribusi Split Data         :\n{df_final['split'].value_counts()}")

# 2. SIMPAN KE CSV UTUH
output_csv_path = BASE_DIR / "processed" / "dataset_metadata_final.csv"
df_final.to_csv(output_csv_path, index=False)

print(f"\n💾 File CSV utuh sukses disimpan di: {output_csv_path}")
print("=========================================================================")